In [1]:
!pip -q install pennylane kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 116.1 MB/s eta 0:00:00


In [2]:
import os
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models, transforms

from PIL import Image

import pennylane as qml
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
PennyLane: 0.45.1
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42

def set_global_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed fixed:", SEED)

Random seed fixed: 42


In [4]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one .pth checkpoint.")

uploaded_filename = next(iter(uploaded))

HQNN_CHECKPOINT = f"/content/{uploaded_filename}"

print("Checkpoint uploaded:")
print(HQNN_CHECKPOINT)

Saving BEST_HQNN (1).pth to BEST_HQNN (1).pth
Checkpoint uploaded:
/content/BEST_HQNN (1).pth


In [5]:
saved_hqnn_state = torch.load(
    HQNN_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

print("Original HQNN checkpoint loaded successfully.")

print(
    "Feature reduction:",
    saved_hqnn_state["feature_reduction.weight"].shape
)

print(
    "Quantum weights:",
    saved_hqnn_state["quantum_layer.weights"].shape
)

print(
    "Classifier:",
    saved_hqnn_state["classifier.weight"].shape
)

Original HQNN checkpoint loaded successfully.
Feature reduction: torch.Size([4, 1280])
Quantum weights: torch.Size([2, 4, 3])
Classifier: torch.Size([4, 4])


In [6]:
path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

all_files = []
all_labels = []


for class_name in classes:

    for folder in ["Training", "Testing"]:

        class_path = (
            Path(path)
            / folder
            / class_name
        )

        for file_path in class_path.iterdir():

            if file_path.is_file():

                all_files.append(
                    str(file_path)
                )

                all_labels.append(
                    class_name
                )


train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)


val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)


print("Total images:", len(all_files))
print("Training:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Testing:", len(test_files), Counter(test_labels))

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Total images: 7200
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [7]:
efficientnet = models.efficientnet_b0(
    weights=None
)

FEATURE_DIM = efficientnet.classifier[1].in_features

assert FEATURE_DIM == 1280


feature_state = {
    key.replace("features.", "", 1): value
    for key, value in saved_hqnn_state.items()
    if key.startswith("features.")
}


efficientnet.features.load_state_dict(
    feature_state,
    strict=True
)


for param in efficientnet.features.parameters():
    param.requires_grad = False


print("EfficientNet-B0 backbone restored successfully.")
print("Feature dimension:", FEATURE_DIM)

print(
    "Trainable EfficientNet parameters:",
    sum(
        p.numel()
        for p in efficientnet.features.parameters()
        if p.requires_grad
    )
)

EfficientNet-B0 backbone restored successfully.
Feature dimension: 1280
Trainable EfficientNet parameters: 0


In [8]:
# ==========================================
# 1-LAYER QUANTUM CIRCUIT
# ==========================================

N_QUBITS = 4
N_Q_LAYERS = 1

quantum_device_1l = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    quantum_device_1l,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit_1l(inputs, weights):

    # Same RY AngleEmbedding
    # No tanh × pi
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # ONE StronglyEntanglingLayer
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # Same Pauli-Z readout
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


weight_shapes_1l = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}


set_global_seed(SEED)

quantum_layer_1l = qml.qnn.TorchLayer(
    quantum_circuit_1l,
    weight_shapes_1l
)


print("1-layer quantum circuit created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)
print("Quantum parameters:", 12)
print("tanh × pi: REMOVED")

1-layer quantum circuit created successfully.
Qubits: 4
Quantum layers: 1
Quantum parameters: 12
tanh × pi: REMOVED


In [9]:
# ==========================================
# 1-LAYER HQNN MODEL
# ==========================================

class HQNNOneLayer(nn.Module):

    def __init__(
        self,
        trained_efficientnet,
        quantum_layer
    ):
        super().__init__()

        # SAME TRAINED EFFICIENTNET BACKBONE
        self.features = trained_efficientnet.features
        self.avgpool = trained_efficientnet.avgpool

        # KEEP BACKBONE FROZEN
        for param in self.features.parameters():
            param.requires_grad = False

        # SAME 1280 -> 4 FEATURE REDUCTION
        self.feature_reduction = nn.Linear(
            FEATURE_DIM,
            N_QUBITS
        )

        # 1-LAYER QUANTUM CIRCUIT
        self.quantum_layer = quantum_layer

        # SAME FINAL CLASSIFIER
        self.classifier = nn.Linear(
            N_QUBITS,
            4
        )


    def forward(self, x):

        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # 1280 -> 4
        x = self.feature_reduction(x)

        # NO tanh × pi

        x = x.cpu()

        x = self.quantum_layer(x)

        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)

        x = self.classifier(x)

        return x


set_global_seed(SEED)

hqnn_1layer = HQNNOneLayer(
    efficientnet,
    quantum_layer_1l
)

print(hqnn_1layer)

HQNNOneLayer(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [10]:
backbone_trainable = sum(
    p.numel()
    for p in hqnn_1layer.features.parameters()
    if p.requires_grad
)

total_trainable = sum(
    p.numel()
    for p in hqnn_1layer.parameters()
    if p.requires_grad
)

print(
    "Trainable EfficientNet parameters:",
    backbone_trainable
)

print(
    "Total trainable parameters:",
    total_trainable
)

print("\nTrainable components:")

for name, parameter in hqnn_1layer.named_parameters():

    if parameter.requires_grad:

        print(
            name,
            tuple(parameter.shape),
            parameter.numel()
        )

Trainable EfficientNet parameters: 0
Total trainable parameters: 5156

Trainable components:
feature_reduction.weight (4, 1280) 5120
feature_reduction.bias (4,) 4
quantum_layer.weights (1, 4, 3) 12
classifier.weight (4, 4) 16
classifier.bias (4,) 4


In [11]:
# ==========================================
# DATASET TRANSFORMS AND DATALOADERS
# ==========================================

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class BrainTumorDataset(Dataset):

    def __init__(self, files, labels, transform=None):

        self.files = files
        self.labels = labels
        self.transform = transform


    def __len__(self):

        return len(self.files)


    def __getitem__(self, idx):

        image = Image.open(
            self.files[idx]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[
            self.labels[idx]
        ]

        return image, label


train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)


BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

Training images: 5040
Validation images: 1080
Testing images: 1080
Batch size: 32


In [12]:
# ==========================================
# EFFICIENTNET FEATURE EXTRACTION
# ==========================================

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Feature extraction device:", feature_device)


# Move only the frozen EfficientNet backbone to GPU
hqnn_1layer.features = hqnn_1layer.features.to(feature_device)
hqnn_1layer.avgpool = hqnn_1layer.avgpool.to(feature_device)

hqnn_1layer.features.eval()
hqnn_1layer.avgpool.eval()


def extract_efficientnet_features(data_loader):

    all_features = []
    all_labels = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(feature_device)

            features = hqnn_1layer.features(images)
            features = hqnn_1layer.avgpool(features)
            features = torch.flatten(features, 1)

            all_features.append(
                features.cpu()
            )

            all_labels.append(
                labels.cpu()
            )

    return (
        torch.cat(all_features, dim=0),
        torch.cat(all_labels, dim=0)
    )


print("Feature extraction function ready.")

Feature extraction device: cuda
Feature extraction function ready.


In [13]:
print("Extracting validation features...")

val_features, val_labels = extract_efficientnet_features(
    val_loader
)

print("Validation feature extraction completed.")
print("Validation features shape:", val_features.shape)
print("Validation labels shape:", val_labels.shape)

Extracting validation features...
Validation feature extraction completed.
Validation features shape: torch.Size([1080, 1280])
Validation labels shape: torch.Size([1080])


In [14]:
# ==========================================
# TRAINING SETUP — 1-LAYER HQNN
# ==========================================

# Keep the trainable HQNN head on CPU
hqnn_1layer.feature_reduction = (
    hqnn_1layer.feature_reduction.cpu()
)

hqnn_1layer.quantum_layer = (
    hqnn_1layer.quantum_layer.cpu()
)

hqnn_1layer.classifier = (
    hqnn_1layer.classifier.cpu()
)


criterion = nn.CrossEntropyLoss()


trainable_parameters = (
    list(hqnn_1layer.feature_reduction.parameters())
    + list(hqnn_1layer.quantum_layer.parameters())
    + list(hqnn_1layer.classifier.parameters())
)


optimizer = torch.optim.Adam(
    trainable_parameters,
    lr=0.001
)


NUM_EPOCHS = 10

best_val_loss = float("inf")
best_epoch = 0


print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Maximum epochs:", NUM_EPOCHS)
print("Quantum layers: 1")
print("EfficientNet backbone trainable: NO")
print("tanh × pi scaling: REMOVED")

Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001
Maximum epochs: 10
Quantum layers: 1
EfficientNet backbone trainable: NO
tanh × pi scaling: REMOVED


In [15]:
# ==========================================
# TRAIN 1-LAYER HQNN
# ==========================================

train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []

training_start_time = time.time()


for epoch in range(NUM_EPOCHS):

    set_global_seed(SEED + epoch)

    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("Extracting augmented training features...")


    # Re-extract training features each epoch
    # so image augmentation remains active

    train_features, train_labels = (
        extract_efficientnet_features(
            train_loader
        )
    )


    train_feature_dataset = TensorDataset(
        train_features,
        train_labels
    )


    train_feature_loader = DataLoader(
        train_feature_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )


    # -------------------------
    # TRAIN
    # -------------------------

    hqnn_1layer.feature_reduction.train()
    hqnn_1layer.quantum_layer.train()
    hqnn_1layer.classifier.train()


    running_train_loss = 0.0

    train_predictions = []
    train_targets = []


    for features, labels in train_feature_loader:

        optimizer.zero_grad()


        # 1280 -> 4
        outputs = (
            hqnn_1layer
            .feature_reduction(features)
        )


        # NO tanh × pi


        # 1-layer quantum circuit
        outputs = (
            hqnn_1layer
            .quantum_layer(outputs)
        )


        # Final 4-class classifier
        outputs = (
            hqnn_1layer
            .classifier(outputs)
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()
        optimizer.step()


        running_train_loss += (
            loss.item()
            * features.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        train_predictions.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
        )

        train_targets.extend(
            labels
            .cpu()
            .numpy()
        )


    epoch_train_loss = (
        running_train_loss
        / len(train_feature_dataset)
    )


    epoch_train_accuracy = accuracy_score(
        train_targets,
        train_predictions
    )


    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )


    # -------------------------
    # VALIDATION
    # -------------------------

    hqnn_1layer.feature_reduction.eval()
    hqnn_1layer.quantum_layer.eval()
    hqnn_1layer.classifier.eval()


    val_feature_dataset = TensorDataset(
        val_features,
        val_labels
    )


    val_feature_loader = DataLoader(
        val_feature_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )


    running_val_loss = 0.0

    val_predictions = []
    val_targets = []


    with torch.no_grad():

        for features, labels in val_feature_loader:

            outputs = (
                hqnn_1layer
                .feature_reduction(features)
            )

            # NO tanh × pi

            outputs = (
                hqnn_1layer
                .quantum_layer(outputs)
            )

            outputs = (
                hqnn_1layer
                .classifier(outputs)
            )


            loss = criterion(
                outputs,
                labels
            )


            running_val_loss += (
                loss.item()
                * features.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            val_predictions.extend(
                predictions
                .cpu()
                .numpy()
            )

            val_targets.extend(
                labels
                .cpu()
                .numpy()
            )


    epoch_val_loss = (
        running_val_loss
        / len(val_feature_dataset)
    )


    epoch_val_accuracy = accuracy_score(
        val_targets,
        val_predictions
    )


    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )


    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)
    train_f1_scores.append(epoch_train_f1)

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)
    val_f1_scores.append(epoch_val_f1)


    # -------------------------
    # SAVE BEST CHECKPOINT
    # -------------------------

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1


        torch.save(
            hqnn_1layer.state_dict(),
            "/content/BEST_HQNN_1_LAYER_NO_TANH_PI.pth"
        )


        marker = " <-- BEST CHECKPOINT"

    else:

        marker = ""


    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Train F1: {epoch_train_f1:.4f}"
    )


    print(
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_accuracy:.4f} | "
        f"Val F1: {epoch_val_f1:.4f}"
        f"{marker}"
    )


training_time = (
    time.time()
    - training_start_time
)


print("\nTRAINING COMPLETED")

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    round(best_val_loss, 4)
)

print(
    "Training time:",
    round(training_time / 60, 2),
    "minutes"
)


EPOCH 1/10
Extracting augmented training features...
Train Loss: 1.1257 | Train Acc: 0.5393 | Train F1: 0.4688
Val Loss: 0.9659 | Val Acc: 0.6667 | Val F1: 0.5970 <-- BEST CHECKPOINT

EPOCH 2/10
Extracting augmented training features...
Train Loss: 0.8132 | Train Acc: 0.8327 | Train F1: 0.8292
Val Loss: 0.7143 | Val Acc: 0.9259 | Val F1: 0.9260 <-- BEST CHECKPOINT

EPOCH 3/10
Extracting augmented training features...
Train Loss: 0.5629 | Train Acc: 0.9595 | Train F1: 0.9595
Val Loss: 0.5049 | Val Acc: 0.9407 | Val F1: 0.9405 <-- BEST CHECKPOINT

EPOCH 4/10
Extracting augmented training features...
Train Loss: 0.4041 | Train Acc: 0.9692 | Train F1: 0.9693
Val Loss: 0.4091 | Val Acc: 0.9389 | Val F1: 0.9386 <-- BEST CHECKPOINT

EPOCH 5/10
Extracting augmented training features...
Train Loss: 0.3068 | Train Acc: 0.9732 | Train F1: 0.9733
Val Loss: 0.3161 | Val Acc: 0.9620 | Val F1: 0.9621 <-- BEST CHECKPOINT

EPOCH 6/10
Extracting augmented training features...
Train Loss: 0.2459 | Train

In [16]:
# ==========================================
# FINAL TEST — 1-LAYER HQNN
# ==========================================

# Load best checkpoint
best_state_1l = torch.load(
    "/content/BEST_HQNN_1_LAYER_NO_TANH_PI.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_1layer.load_state_dict(
    best_state_1l
)

print("Best 1-layer HQNN checkpoint loaded successfully.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)


# ------------------------------------------
# EXTRACT TEST FEATURES
# ------------------------------------------

print("\nExtracting test features...")

test_features, test_labels = (
    extract_efficientnet_features(
        test_loader
    )
)

print("Test feature extraction completed.")
print("Test features shape:", test_features.shape)
print("Test labels shape:", test_labels.shape)


# ------------------------------------------
# TEST FEATURE LOADER
# ------------------------------------------

test_feature_dataset = TensorDataset(
    test_features,
    test_labels
)

test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------
# EVALUATION MODE
# ------------------------------------------

hqnn_1layer.feature_reduction.eval()
hqnn_1layer.quantum_layer.eval()
hqnn_1layer.classifier.eval()


test_targets = []
test_predictions = []
test_probabilities = []

running_test_loss = 0.0


# ------------------------------------------
# TEST
# ------------------------------------------

with torch.no_grad():

    for features, labels in test_feature_loader:

        # 1280 -> 4
        outputs = (
            hqnn_1layer
            .feature_reduction(features)
        )

        # NO tanh × pi

        # 1-layer quantum circuit
        outputs = (
            hqnn_1layer
            .quantum_layer(outputs)
        )

        # Final classifier
        outputs = (
            hqnn_1layer
            .classifier(outputs)
        )

        loss = criterion(
            outputs,
            labels
        )

        running_test_loss += (
            loss.item()
            * features.size(0)
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_targets.extend(
            labels.cpu().numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# ------------------------------------------
# NUMPY
# ------------------------------------------

test_targets = np.array(
    test_targets
)

test_predictions = np.array(
    test_predictions
)

test_probabilities = np.array(
    test_probabilities
)


# ------------------------------------------
# METRICS
# ------------------------------------------

test_loss = (
    running_test_loss
    / len(test_feature_dataset)
)

test_accuracy = accuracy_score(
    test_targets,
    test_predictions
)

test_precision = precision_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_recall = recall_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_f1 = f1_score(
    test_targets,
    test_predictions,
    average="macro"
)

test_auc = roc_auc_score(
    test_targets,
    test_probabilities,
    multi_class="ovr",
    average="macro"
)


# ------------------------------------------
# RESULTS
# ------------------------------------------

print(
    "\nFINAL 1-LAYER HQNN TEST RESULTS"
)

print(
    f"Test Loss:       {test_loss:.4f}"
)

print(
    f"Accuracy:        "
    f"{test_accuracy:.4f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Macro Precision: "
    f"{test_precision:.4f} "
    f"({test_precision * 100:.2f}%)"
)

print(
    f"Macro Recall:    "
    f"{test_recall:.4f} "
    f"({test_recall * 100:.2f}%)"
)

print(
    f"Macro F1:        "
    f"{test_f1:.4f} "
    f"({test_f1 * 100:.2f}%)"
)

print(
    f"Macro ROC-AUC:   "
    f"{test_auc:.4f} "
    f"({test_auc * 100:.2f}%)"
)


print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        test_targets,
        test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)


print("\nCONFUSION MATRIX")

print(
    confusion_matrix(
        test_targets,
        test_predictions
    )
)

Best 1-layer HQNN checkpoint loaded successfully.
Best epoch: 10
Best validation loss: 0.18307848158809875

Extracting test features...
Test feature extraction completed.
Test features shape: torch.Size([1080, 1280])
Test labels shape: torch.Size([1080])

FINAL 1-LAYER HQNN TEST RESULTS
Test Loss:       0.2224
Accuracy:        0.9491 (94.91%)
Macro Precision: 0.9490 (94.90%)
Macro Recall:    0.9491 (94.91%)
Macro F1:        0.9488 (94.88%)
Macro ROC-AUC:   0.9916 (99.16%)

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Glioma     0.9455    0.9000    0.9222       270
  Meningioma     0.9051    0.9185    0.9118       270
    No Tumor     0.9710    0.9926    0.9817       270
   Pituitary     0.9744    0.9852    0.9797       270

    accuracy                         0.9491      1080
   macro avg     0.9490    0.9491    0.9488      1080
weighted avg     0.9490    0.9491    0.9488      1080


CONFUSION MATRIX
[[243  23   3   1]
 [ 12 248   5   5]
 [  1   